In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import numpy as np
import os
import random
import glob
import matplotlib.pyplot as plt
import sys
from google.colab import drive
drive.mount("/content/drive")
!cp "/content/drive/MyDrive/data2.zip" /content/
!unzip -q /content/data2.zip -d /content/
!ls /content/

def compute_gap_mask(t):
    threshold = 50
    dt = np.diff(t)
    g_mask = dt <= threshold
    g_mask = np.concatenate([[True], g_mask])
    return g_mask

def parse_params(path):
    wanted = ["Tmax", "tau", "umin", "fbl", "I_bl"]
    values = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2 and parts[0] in wanted:
                values[parts[0]] = float(parts[1])
    return np.array([values[k] for k in wanted])

class MicrolensingDataset(Dataset):
    def __init__(self, file_paths, file_paths_params, stats=None):
        self.file_paths = []
        self.params_paths = []
        self.param_list = []
        self.curve_list = []
        self.first_times = []
        self.i_bl_guesses = []
        self.event_end_idxs = []

        for curve, param in zip(file_paths, file_paths_params):
            cv = np.loadtxt(curve, comments=["#", "col"], usecols=(0,1,2))
            pm = parse_params(param)
            I_bl_guess = np.median(cv[0:20, 1])
            mag_std = np.std(cv[0:20, 1])

            peak_global_idx = int(np.argmin(cv[:, 1]))

            # event_start: search BACKWARD from peak for 5 consecutive baseline points
            num_in_baseline = 0
            event_start_idx = -1
            for i in range(peak_global_idx - 1, -1, -1):
                if cv[i, 1] > I_bl_guess - 2 * mag_std:
                    num_in_baseline += 1
                else:
                    num_in_baseline = 0
                if num_in_baseline == 5:
                    event_start_idx = i + 4
                    break
            # fallback: forward search from start of curve
            if event_start_idx == -1:
                num_over_std = 0
                for i in range(len(cv)):
                    if cv[i,1] <= I_bl_guess - 2 * mag_std:
                        num_over_std += 1
                    else:
                        num_over_std = 0
                    if num_over_std == 5:
                        event_start_idx = i - 4
                        break
            if event_start_idx == -1:
                continue

            # event_end: forward search, 5 consecutive baseline points after peak
            num_in_baseline = 0
            event_end_global_idx = len(cv) - 1
            for i in range(peak_global_idx + 1, len(cv)):
                if cv[i, 1] > I_bl_guess - 2 * mag_std:
                    num_in_baseline += 1
                else:
                    num_in_baseline = 0
                if num_in_baseline == 5:
                    event_end_global_idx = i - 4
                    break

            trim_start = max(0, event_start_idx - 50)
            trim_end = min(len(cv), event_end_global_idx + 50)
            if trim_end - trim_start < 25 or event_end_global_idx <= event_start_idx:
                continue
            cv = cv[trim_start:trim_end, :]
            event_end_local = event_end_global_idx - trim_start

            first_time = cv[0,0]
            cv[:, 0] -= first_time
            cv[:, 1] -= I_bl_guess
            pm[0] -= first_time
            pm[4] -= I_bl_guess

            self.param_list.append(pm)
            self.curve_list.append(cv)
            self.first_times.append(first_time)
            self.i_bl_guesses.append(I_bl_guess)
            self.file_paths.append(curve)
            self.params_paths.append(param)
            self.event_end_idxs.append(event_end_local)

        self.param_list = np.stack(self.param_list)
        if stats:
            self.pmeans = stats[0]
            self.pstds = stats[1]
        else:
            self.pmeans = np.mean(self.param_list, axis=0)
            self.pstds = np.std(self.param_list, axis=0)
        self.param_list = (self.param_list - self.pmeans)/self.pstds


    def __len__(self):
        return len(self.curve_list)

    def __getitem__(self, idx):
        lc = self.curve_list[idx]
        pms = self.param_list[idx]
        total_length = len(lc)

        peak_idx = int(np.argmin(lc[:, 1]))
        event_end_idx = self.event_end_idxs[idx]

        # magnitude-uniform cutoff within active descent
        if peak_idx + 1 >= event_end_idx:
            cutoff_idx = total_length - 1
        else:
            peak_mag = lc[peak_idx, 1]
            baseline_mag = lc[event_end_idx, 1]
            target_mag = random.uniform(peak_mag, baseline_mag)
            cutoff_idx = event_end_idx
            for i in range(peak_idx + 1, event_end_idx + 1):
                if lc[i, 1] >= target_mag:
                    cutoff_idx = i
                    break

        X = lc[:cutoff_idx, :]
        y = pms[:]

        time_steps = X[:, 0]
        numpy_gap_mask = compute_gap_mask(time_steps)

        X_tensor = torch.tensor(X, dtype=torch.float32)
        y_tensor = torch.tensor(y, dtype=torch.float32)
        X_length = X_tensor.shape[0]
        ft_tensor = torch.tensor(self.first_times[idx], dtype=torch.float32)
        ibl_tensor = torch.tensor(self.i_bl_guesses[idx], dtype=torch.float32)
        gap_mask_tensor = torch.tensor(numpy_gap_mask, dtype=torch.bool)

        return X_tensor, y_tensor, X_length, gap_mask_tensor, ft_tensor, ibl_tensor

def pad_collate(batch):
    x_list = [item[0] for item in batch]
    y_list = [item[1] for item in batch]
    x_lengths = [item[2] for item in batch]
    ft_tensors = [item[4] for item in batch]
    ibl_tensors = [item[5] for item in batch]

    x_padded = pad_sequence(x_list, batch_first=True, padding_value=0.0)
    y_stacked = torch.stack(y_list)
    ft_stacked = torch.stack(ft_tensors)
    ibl_stacked = torch.stack(ibl_tensors)
    x_lengths_tensor = torch.tensor(x_lengths, dtype=torch.int64)

    gap_masks_list = [item[3] for item in batch]
    gap_masks_padded = pad_sequence(gap_masks_list, batch_first=True, padding_value=False)

    return x_padded, y_stacked, x_lengths_tensor, gap_masks_padded, ft_stacked, ibl_stacked

def load_data(filepath, stats=None):
    curves_file_path = f"/content/{filepath}/**/curve_*.dat"
    params_file_path = f"/content/{filepath}/**/params_*.dat"
    lightcurves = glob.glob(curves_file_path, recursive=True)
    paramcurves = glob.glob(params_file_path, recursive=True)

    params_dict = {}
    for p in paramcurves:
        name = os.path.basename(p)
        key = name.replace("params_", "").replace(".dat", "")
        params_dict[key] = p

    matched_curves = []
    matched_params = []
    for c in lightcurves:
        name = os.path.basename(c)
        key = name.replace("curve_", "").replace(".dat", "")
        if key in params_dict:
            matched_curves.append(c)
            matched_params.append(params_dict[key])

    print(f"Successfully matched {len(matched_curves)} curve-params pairs!")

    if len(matched_curves) > 0:
        dataset = MicrolensingDataset(matched_curves, matched_params, stats=stats)
        dataloader = DataLoader(dataset, batch_size=32, collate_fn=pad_collate, shuffle=True)
    return dataset, dataloader


In [ ]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence
import os

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=3, hidden_size=256,num_layers=3, dropout=0.2)
        self.fc1 = nn.Linear(256,64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64,5)

    def forward(self, x):
        out, (hidden, cell) = self.lstm(x)
        out = self.fc1(hidden[-1])
        out = self.relu(out)
        out = self.fc2(out)
        return out

model = Model()
train_dataset, train_dataloader = load_data("training 3")
test_dataset, test_dataloader = load_data("testing 3", [train_dataset.pmeans, train_dataset.pstds])
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
best_test_loss = float("inf")
real_params = []
pred_params = []
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using", device)
model = model.to(device)
param_weights = torch.tensor([1.0,1.0,1.0,1.0,1.0], device=device)

    #train
start_epoch = 0
ckpt_path = "/content/drive/MyDrive/checkpoint.pt"
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    best_test_loss = ckpt['best_test_loss']
    start_epoch = ckpt['epoch'] + 1
    print(f"Resuming from epoch {start_epoch}, best_test_loss={best_test_loss:.4f}")
else:
    print("No checkpoint found, starting fresh")
for epoch in range(start_epoch,300):
    model.train()
    epoch_loss = 0
    n_batches = 0
    for X_padded, y_stacked, lengths, gap_masks, ft_stacked, ibl_stacked in train_dataloader:
        X_padded = X_padded.to(device)
        y_stacked = y_stacked.to(device)
        X = pack_padded_sequence(X_padded, lengths, batch_first = True, enforce_sorted=False)
        pred = model(X)
        loss = (((pred-y_stacked) ** 2) * param_weights).mean()
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    model.eval()
    test_loss = 0
    test_batches = 0
    with torch.no_grad():
        for X_padded, y_stacked, lengths, gap_masks, ft_stacked, ibl_stacked in test_dataloader:
            X_padded = X_padded.to(device)
            y_stacked = y_stacked.to(device)
            X = pack_padded_sequence(X_padded, lengths, batch_first=True, enforce_sorted=False)
            pred = model(X)
            test_loss += (((pred - y_stacked) ** 2) * param_weights).mean().item()
            test_batches += 1
    avg_test = test_loss/test_batches
    print(f"Epoch {epoch}: train = {epoch_loss/n_batches:.4f}, test = {test_loss/test_batches:.4f}")
    if avg_test < best_test_loss:
          best_test_loss = avg_test
          torch.save(model.state_dict(), "/content/drive/MyDrive/best_model.pt")
          print(f"  saved new best (test = {avg_test:.4f})")
    torch.save({
      'epoch': epoch,
      'model': model.state_dict(),
      'optimizer': optimizer.state_dict(),
      'best_test_loss': best_test_loss,
    }, "/content/drive/MyDrive/checkpoint.pt")


    #test
model.load_state_dict(torch.load("/content/drive/MyDrive/best_model.pt", map_location=device))
model.eval()
total_loss = 0
batch_count = 0
pmeans_t = torch.tensor(test_dataset.pmeans, dtype=torch.float32)
pstds_t = torch.tensor(test_dataset.pstds, dtype=torch.float32)
per_param_sq = torch.zeros(5)

with torch.no_grad():
    for X_padded, y_stacked, lengths, gap_masks, ft_stacked, ibl_stacked in test_dataloader:
        X_padded = X_padded.to(device)
        y_stacked = y_stacked.to(device)
        X = pack_padded_sequence(X_padded, lengths, batch_first = True, enforce_sorted=False)
        y_stacked = y_stacked.cpu()
        pred = model(X).cpu()
        loss = ((pred-y_stacked) ** 2).mean()
        total_loss += loss.item()
        per_param_sq += ((pred-y_stacked)**2).mean(dim=0)
        batch_count += 1
    pred_real = pred * pstds_t + pmeans_t
    pred_real[:, 0] += ft_stacked
    pred_real[:, 4] += ibl_stacked
    true_real = y_stacked * pstds_t + pmeans_t
    true_real[:, 0] += ft_stacked
    true_real[:, 4] += ibl_stacked

    for i in range(len(pred_real)):
        print(f"  pred: {pred_real[i].numpy()}")
        pred_params.append(pred_real[i].numpy())
        print(f"  true: {true_real[i].numpy()}")
        real_params.append(true_real[i].numpy())
        print()


print("Test loss (avg):", total_loss/batch_count)
print("Per-param MSE:", per_param_sq/batch_count)



Successfully matched 3810 curve-params pairs!
Successfully matched 673 curve-params pairs!
Using cuda
Resuming from epoch 72, best_test_loss=0.4085
Epoch 72: train = 0.4671, test = 0.4245
Epoch 73: train = 0.4701, test = 0.4145
Epoch 74: train = 0.4704, test = 0.4168
Epoch 75: train = 0.4727, test = 0.3883
  saved new best (test = 0.3883)
Epoch 76: train = 0.4512, test = 0.3919
Epoch 77: train = 0.4623, test = 0.4011
Epoch 78: train = 0.4544, test = 0.3939
Epoch 79: train = 0.4647, test = 0.4065
Epoch 80: train = 0.4643, test = 0.4272
Epoch 81: train = 0.4511, test = 0.3881
  saved new best (test = 0.3881)
Epoch 82: train = 0.4454, test = 0.3825
  saved new best (test = 0.3825)
Epoch 83: train = 0.4506, test = 0.3884
Epoch 84: train = 0.4537, test = 0.3980
Epoch 85: train = 0.4444, test = 0.3936
Epoch 86: train = 0.4315, test = 0.3937
Epoch 87: train = 0.4301, test = 0.3891
Epoch 88: train = 0.4374, test = 0.3970
Epoch 89: train = 0.4384, test = 0.3842
Epoch 90: train = 0.4487, test = 